# BriskVLA evaluation on Colab

Runs `project/briskvla/run_eval.sh` for one model, the same script used on the RTX 3080, so the
settings are identical. Results are saved to Google Drive, and finished suites are skipped, so after a
disconnect you can just run all cells again.

1. *Runtime → Change runtime type →* GPU.
2. Edit the **Config** cell and run all cells.

Latency is reported on the RTX 3080, so it is off here by default.

## Config

In [ ]:
BRANCH = "vyvas/student-expert"

NAME = "teacher"      # name of the output folders
POLICY = "teacher"    # "teacher" or a checkpoint folder, e.g. "/content/students/k8_pairs"

STEPS = "10 3"
SUITES = "libero_spatial libero_object libero_goal libero_10"
EPISODES = 10         # per task
SUCCESS = 1           # 0 to skip the LIBERO success evals
LATENCY = 0           # 1 to also measure latency on this GPU

OUT = "/content/drive/MyDrive/briskvla/outputs"
REPO_URL = "https://github.com/vyvas33/briskvla.git"
REPO_DIR = "/content/briskvla"

## 1. GPU and Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

from google.colab import drive

drive.mount("/content/drive")

## 2. Install

In [ ]:
import os
import shutil

!apt-get -qq install -y libegl1 libgl1 ffmpeg > /dev/null
if shutil.which("uv") is None:
    !pip install -q uv

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

# Colab's own Python is newer than what the repo is tested with, so pin 3.12
!cd {REPO_DIR} && uv sync --locked --inexact --python 3.12 --extra smolvla --extra libero

## 3. Optional: build an untrained student

Only needed to evaluate a student right after copying the teacher's layers. Set `POLICY` in the Config
cell to the folder below and give it its own `NAME`, e.g. `k8_pairs_untrained`.

In [ ]:
STUDENT_LAYERS = "0,1,4,5,8,9,12,13"
STUDENT_DIR = "/content/students/k8_pairs"

!cd {REPO_DIR} && uv run --no-sync python project/briskvla/build_student.py --layers {STUDENT_LAYERS} --out {STUDENT_DIR}

## 4. Run

In [ ]:
os.environ.update({
    "STEPS": STEPS,
    "SUITES": SUITES,
    "EPISODES": str(EPISODES),
    "SUCCESS": str(SUCCESS),
    "LATENCY": str(LATENCY),
    "OUT": OUT,
    "CKPT": "/content/checkpoints",
    "PYTHON": "3.12",
})
!cd {REPO_DIR} && bash project/briskvla/run_eval.sh {NAME} {POLICY}

## 5. Summary of everything in Drive

In [ ]:
!cd {REPO_DIR} && uv run --no-sync python project/briskvla/summarize_eval.py --root {OUT}/eval